# Indexation avancée

## Gestion des doublons

Jusqu'à présent nous identifions les doublons grâce à leur hash mais cette méthode à plusieurs limites:
- Deux documents avec une extension différente donnent des hash différents
- Deux documents avec ne serait-ce qu'un retour à la ligne de différence donne un hash différents
- On ne gère pas les chunks redondant. "Politique de télétravail 2024" vs "Politique de télétravail 2025" peuvent très bien n'avoir qu'une différence minimale comme par exemple passer de 3 à 2 jours de télétravail et le reste est inchangé

Comme à chaque fois, il faut joueur sur tous les tableaux

### Normaliser le texte

L'option la plus simple est de normaliser les textes en ne gardant que les espaces et les mots. Attention que dans certains cas cela peut poser problèmes et il faut un regex affiné.

In [1]:
import polars as pl

import regex as re
from unidecode import unidecode

from datasketch import MinHash, MinHashLSH

from openai import OpenAI
from qdrant_client import QdrantClient, models

In [2]:
EMBEDDING_URL = 'http://100.85.184.14:8000/v1'
EMBEDDING_MODEL = 'bge-m3'
MAX_TOKENS = 512

QDRANT_URL = 'http://localhost:6333'
QDRANT_API_KEY = '1234'
COLLECTION = 'Veltis'


qdrant = QdrantClient(url=QDRANT_URL, api_key=QDRANT_API_KEY)
embedder = OpenAI(base_url=EMBEDDING_URL, api_key='no-sk')

C:\Users\romainwauman\AppData\Local\Temp\ipykernel_67596\3413583175.py:10: UserWarning: Api key is used with an insecure connection.
  qdrant = QdrantClient(url=QDRANT_URL, api_key=QDRANT_API_KEY)


In [3]:
def normalize(text, unaccent = True):
    if unaccent:
        return ' '.join(re.findall(r'\w+', unidecode(text.lower())))
    return ' '.join(re.findall(r'\w+', text.lower()))

text1 = "L'éléphant est dans la pièce.\nIl marche sur des oeufs."
text2 = "L elephant est dans la pièce. IL marche sur des œufs."

normalize(text1) == normalize(text2)

True

### Les quasis doublons

#### Similarité de Jaccard

Cette méthode ne permet pas de tout capturer pour autant. Des phrases très proches ou reformulées passent au travers de ce premier filtre.

Une des solutions consiste à découper nos textes en tout petit morceaux(shlings/tuiles), beaucoup plus petits que des chunks et de calculer une similarité de Jaccard. On appelle cela des tuiles parce que les textes se chevauchent comme une fenêtre glissante, ce n'est pas une partition.

```Ceci est une tuile``` renvoie pour une taille 2 de tuile:
> 'Ceci est'
> 
> 'est une'
> 
> 'une tuile'

Ensuite on divise le nombre totale de tuile unique (union - ``&``) par le nombre de tuiles uniques communes (intersection -  ``|``)

Quand elle vaut 1, les deux textes sont identiques et toutes les tuiles sont partagées. Avec des valeurs proches de 0.9, on a affaire à des quasi-doublons mais attention les détails peuvent se cacher dans ces 0.10 restants.

| Similarité | Interprétation habituelle | Que faire |
|---|---|---|
| Texte identique (même empreinte de texte) | Doublon exact | Ne garder qu'un exemplaire |
| Au-dessus de 0,9 environ | Quasi-doublon : même contenu, autre format ou autre mise en page | Ne garder que la meilleure source |
| Entre 0,5 et 0,9 environ | Une autre version, ou un document qui en reprend largement un autre | Lire les deux documents, puis décider |
| En dessous | Documents différents | Rien |

In [4]:
def shingles(text, n=2):
    normalized_text = normalize(text)
    normalized_text = normalized_text.split()
    return {' '.join(normalized_text[i:i+n]) for i in range(len(normalized_text) - n + 1)}

def jaccard(u, v):
    if u | v:
        return len(u & v)/len(u | v)
    return 0.0

text1 = "Nous autorisons deux jours de télétravail par semaine."
text2 = "Nous autorisons le télétravail deux jours par semaine."

shingles_text1 = shingles(text1)
shingles_text2 = shingles(text2)
jaccard(shingles_text1, shingles_text2)

0.2727272727272727

Dans ce cas la valeur peut sembler petite malgré le fait que les deux textes soit très proches. C'est dû au fait que le texte soit petit, sur des documents beaucoup plus grands, cette méthode fonctionne bien.

Elle a cependant un défaut majeur car elle implique de comparer les documents 2 à 2 pour trouver les documents redondants.

Avec 100 documents, cela donne (100 * (100-1)) = 9900 comparaisons

Avec 50000 documents cela donne 50_000 * 49_000 = 2_499_950_000

#### MinHash et Locality-Sensitive Hashing (LSH)

MinHash résume chaque texte, ou plus précisément son ensemble de shingles, en une empreinte de taille fixe (par exemple 128 valeurs), quelle que soit la longueur du texte. Le Jaccard est ensuite estimé à partir de ces empreintes au lieu d'être calculé exactement, c'est une approximation. 

À elle seule, cette méthode ne résout pas notre problème. Elle réduit la mémoire et le coût de chaque comparaison, mais il nous faut toujours comparer toutes les paires. Sa propriété clé est que deux textes similaires ont une forte probabilité d'avoir les mêmes valeurs dans leur empreinte : la probabilité que deux valeurs à la même position soient identiques est égale au Jaccard des deux textes. C'est ce qui rend possible la seconde étape.

LSH découpe ensuite chaque empreinte en bandes et range les documents dans des *buckets* et deux documents qui partagent au moins une bande identique deviennent candidats. On n'a donc plus à comparer les n(n-1)/2 paires possibles, mais seulement les candidats, c'est-à-dire les documents probablement similaires. Comme la méthode est probabiliste, quelques paires similaires peuvent être manquées et quelques paires éloignées retenues. On vérifie donc que le score des candidats retenus.

In [ ]:
from datasketch import MinHash, MinHashLSH
m1, m2 = MinHash.bulk(
    [[s.encode('utf-8') for s in shingles_text1],
    [s.encode('utf-8') for s in shingles_text2]],
    num_perm = 128
)
print(m1.hashvalues)
m1.jaccard(m2)


[ 209088744 1945912085  466877472   46126977  427019063  904560446
  395956225  162187834  177306977  906337134  534563592 1083614914
   31851047  714500146  741855857   35228315  453541915 1038202268
  269096677  660131846  564277172 1178365015  351515870   52075085
  188143999   82326708  129715313  788325295  147147914 1266863455
 1682542756  406655786 1289354716   68938580  432359345   45625727
  659446185   56702504  574424917   36958820  399123444  207124081
  118672900  512919449  306392979  951531613  210352408  716264649
 1100103602    5174128 1886816713 1012292249  490424252 1375150596
  621185311 1380274055  144503831 1589632382  505830505    9518009
  374274865 1664116910 2401126867  371461945  305408167  474926559
  268498489   40805326  515733343  265334312  897147889  668496730
  555698484  153817478  429042384   40604470  181007654   71528253
  110863441  587651203  137606404  314702386  409928356  671315667
  123698883  112469846 1547084802  203153237  574603872  85512

0.28125

In [ ]:
def minhash(text, n = 2, num_perm = 128):
    m = MinHash(num_perm = num_perm)
    m.update_batch([s.encode('utf-8') for s in shingles(text, n)])
    return m

docs = {
    'd1': text1,
    'd2': text2,
    'd3': 'Le télétravail est autorisé deux jours par semaine.',
    'd4': 'La cantine ferme à 14 heures le vendredi.',
}
signatures = {key: minhash(text) for key, text in docs.items()}

lsh = MinHashLSH(threshold = 0.2, num_perm = 128)
for key, m in signatures.items():
    lsh.insert(key, m)

for key, m in signatures.items():
    candidats = [c for c in lsh.query(m) if c != key]
    scores = {c: round(m.jaccard(signatures[c]), 2) for c in candidats}
    print(f'{key}: {scores}')

d1: {'d3': 0.21, 'd2': 0.28}
d2: {'d3': 0.41, 'd1': 0.28}
d3: {'d1': 0.21, 'd2': 0.41}
d4: {}


### Maximal Marginal Relevance (MMR)

Les techniques précédentes permettent de faire un premier gros nettoyage et ne suffisent pas toujours. Surtout elles ne s'attaquent jamais à la question de la sémantique. Deux textes lexilament différents peuvent vouloir dire la même chose.

Il est cependant possible lors de la récupération des chunks d'inclure une notion de diversité.

Le MMR permet de placer un curseur de compromis entre la redonance et la pertinence. C'est un algorithme itératif qui introduit une petite latence puisque chaque candidat doit être sélectionné un à un et évalué par rapport à ceux déjà sélectionnés. C'est en réalité assez négligeable sur un top 100.
Notez quelques points:
- Il s'agit d'un algorthime glouton donc il optimise localement la récupération. C'est important parce que les postérieurs n'ont pas d'influence, on ne réordonne pas toute la sélection à chaque chunk sélectionné.
- La pénalité se traduit pas une valeure comprise entre 0 et 1. 0 pour maximiser la diversité mais on ne tient presque plus compte de la requête et 1 afin de maximiser la pertinence. Cette valeur est déterminée empiriquement et dépend entièrement du contexte. Dans la pratique, on préconise une valeur entre 0.5 et 0.7.
- L'embedding peut jouer un grand rôle. Si l'embedding est mauvais, cette notion de diversité peut être faussée.

#### L'utilisation dans Qdrant

Le MMR seul peut être utilisé pour récupérer des chunks mais alors vous excluez les vecteurs sparse nécessairement.
Pour combinaison recherche dense, sparse et MMR il faut le faire en plusieurs étape:
1. Recherche Sparse + Dense qui chacune renvoie x chunks
2. On garde le meilleur des deux via RRF
3. Sur ces meilleurs on applique le MMR afin de gagner en diversité.

Deux poins importants:
1. Faire le MMR avant RRF n'a pas trop de sens parce que le RRF pourrait à nouveau reintroduire des vecteurs redondants
2. Votre première requête doit ramener plus que ce que le deuxième renvoie


In [7]:
BM25_OPTIONS = {
    'language': 'french',
    'ascii_folding': True,
    'avg_len': 50
    }

def embed(texts, client, batch_size=32):
    vectors = []
    for start in range(0, len(texts), batch_size):
        batch = texts[start:start+batch_size]
        r = client.embeddings.create(model=EMBEDDING_MODEL, input=batch)
        vectors.extend(vector.embedding for vector in r.data)
    return vectors

def get_bm25(text):
    return models.Document(text=text, model='qdrant/bm25', options=BM25_OPTIONS)

def search_hybrid(text, qdrant_client, embed_client, limit=10, search_limit=20, query_filter=None):
    return qdrant_client.query_points(
        COLLECTION,
        prefetch = [
            models.Prefetch(query=embed([text], embed_client)[0], using='dense', limit=search_limit),
            models.Prefetch(query=get_bm25(text), using='sparse', limit=search_limit)
        ],
        query = models.RrfQuery(rrf=models.Rrf(k=60)),
        limit=limit,
        query_filter=query_filter
    ).points

def search_hybrid_with_mmr(text, qdrant_client, embed_client, limit=10, search_limit=20, query_filter=None, diversity = 0.5):
    query_embedding = embed([text], embed_client)[0]
    return qdrant_client.query_points(
        COLLECTION,
        prefetch = models.Prefetch(
            prefetch = [
                models.Prefetch(query = query_embedding, using='dense', limit=search_limit),
                models.Prefetch(query = get_bm25(text), using='sparse', limit=search_limit)
            ],
            query=models.RrfQuery(rrf=models.Rrf(k=60)),
            limit=search_limit
        ),
        query = models.NearestQuery(
            nearest = query_embedding,
            mmr = models.Mmr(diversity = diversity, candidates_limit = search_limit)
        ),
        using = 'dense',
        limit = limit,
        query_filter = query_filter
    ).points

query = 'Combien de Veltis Trike ont été produits ?'

display(pl.from_dicts(search_hybrid(query, qdrant, embedder, limit=3)).unnest('payload').sort('score', descending=True))

pl.from_dicts(search_hybrid_with_mmr(query, qdrant, embedder, limit=3, diversity=0.6)).unnest('payload').sort('score', descending=True)


id,version,score,document,chunk_index,headings,text,file_extension,vector,shard_key,order_value
str,i64,f64,str,i64,str,str,str,null,null,null
"""a63a356c-125d-553f-8c0f-427c07…",3,0.033333,"""intranet_actualites.html""",4,"""Actualités | Le 1 000e Veltis …","""Actualités Le 1 000e Veltis Tr…","""html""",null,null,null
"""5a2b8efe-48a1-56bb-8fbe-0080b5…",3,0.032266,"""newsletter_le_fil_2026-02.pdf""",2,"""Portrait : Karim Benali, douze…","""Portrait : Karim Benali, douze…","""pdf""",null,null,null
"""31dacfe5-5939-53a2-9592-108b96…",3,0.032002,"""rapport_annuel_2025.pdf""",4,"""4. Ventes par modèle et par tr…","""4. Ventes par modèle et par tr…","""pdf""",null,null,null


id,version,score,document,chunk_index,headings,text,file_extension,vector,shard_key,order_value
str,i64,f64,str,i64,str,str,str,null,null,null
"""a63a356c-125d-553f-8c0f-427c07…",3,0.679959,"""intranet_actualites.html""",4,"""Actualités | Le 1 000e Veltis …","""Actualités Le 1 000e Veltis Tr…","""html""",null,null,null
"""5a2b8efe-48a1-56bb-8fbe-0080b5…",3,0.5734801,"""newsletter_le_fil_2026-02.pdf""",2,"""Portrait : Karim Benali, douze…","""Portrait : Karim Benali, douze…","""pdf""",null,null,null
"""26581f23-00e8-56ff-9ca0-414f29…",3,0.4708476,"""presentation_veltis.pdf""",3,"""Nos valeurs""","""Nos valeurs - Robustesse : un …","""pdf""",null,null,null


In [8]:
display(pl.from_dicts(search_hybrid_with_mmr(query, qdrant, embedder, limit=3, diversity=1)).unnest('payload').sort('score', descending=True))


pl.from_dicts(search_hybrid_with_mmr(query, qdrant, embedder, limit=3, diversity=0)).unnest('payload').sort('score', descending=True)


id,version,score,document,chunk_index,headings,text,file_extension,vector,shard_key,order_value
str,i64,f64,str,i64,str,str,str,null,null,null
"""a63a356c-125d-553f-8c0f-427c07…",3,0.679959,"""intranet_actualites.html""",4,"""Actualités | Le 1 000e Veltis …","""Actualités Le 1 000e Veltis Tr…","""html""",null,null,null
"""39c21d64-4996-511f-8d2a-195fd4…",3,0.279194,"""newsletter_le_fil_2026-02.pdf""",0,"""Le Fil Veltis""","""Le Fil Veltis La lettre intern…","""pdf""",null,null,null
"""da8fd498-3737-56a5-a7cd-68f5d5…",3,0.184794,"""intranet_faq_rh.html""",1,"""FAQ Ressources humaines | Cong…","""FAQ Ressources humaines Congés…","""html""",null,null,null


id,version,score,document,chunk_index,headings,text,file_extension,vector,shard_key,order_value
str,i64,f64,str,i64,str,str,str,null,null,null
"""a63a356c-125d-553f-8c0f-427c07…",3,0.679959,"""intranet_actualites.html""",4,"""Actualités | Le 1 000e Veltis …","""Actualités Le 1 000e Veltis Tr…","""html""",null,null,null
"""83f0b8c4-d6be-5789-8a8d-e4a472…",3,0.584353,"""presentation_veltis.pdf""",8,"""Dates clés""","""Dates clés - 2011 : création d…","""pdf""",null,null,null
"""31dacfe5-5939-53a2-9592-108b96…",3,0.5815892,"""rapport_annuel_2025.pdf""",4,"""4. Ventes par modèle et par tr…","""4. Ventes par modèle et par tr…","""pdf""",null,null,null
